# NB01b — Pinpoint ROI Full-Frame

ROI dipakai sebagai sandbox deteksi, bukan sebagai koordinat output. Polygon harus mencakup seluruh lintasan kendaraan dan diberi margin kecil agar kendaraan tidak terpotong. Hasil disimpan dalam koordinat resolusi asli setiap kamera.

Kontrol: klik kiri pertama memulai polygon baru, klik kiri berikutnya menambah titik, klik kanan atau `u` menghapus titik terakhir, `Enter` langsung menyimpan, `Esc` mempertahankan ROI lama.

In [1]:
%matplotlib tk
import csv
import importlib
import json
import os
import sys
from pathlib import Path

from PIL import Image

RISET_ROOT = Path(os.environ.get("RISET_PROJECT_ROOT", Path.cwd())).resolve()
if RISET_ROOT.name not in {"Riset-HampirMatang", "NewStyle"}:
    raise RuntimeError(
        "Jalankan notebook dari root proyek atau set RISET_PROJECT_ROOT."
    )
sys.path.insert(0, str(RISET_ROOT / "builders"))
import build_roi_config

build_roi_config = importlib.reload(build_roi_config)
from build_roi_config import load_roi_config, pick_polygon_for_camera, save_roi_config

MANIFEST_PATH = RISET_ROOT / "output" / "manifests" / "frames_manifest.csv"
ROI_PATH = RISET_ROOT / "output" / "roi" / "roi_config.json"
VIDEO_DIR = RISET_ROOT / "output" / "videos"
FRAMES_DIR = RISET_ROOT / "output" / "frames"


def local_path(value):
    path = Path(value)
    if path.exists():
        return path
    if "output" in path.parts:
        relative_candidate = RISET_ROOT / Path(
            *path.parts[path.parts.index("output") :]
        )
        if relative_candidate.exists():
            return relative_candidate
    candidates = list(
        (VIDEO_DIR if path.suffix.lower() == ".mp4" else FRAMES_DIR).rglob(path.name)
    )
    if len(candidates) != 1:
        raise FileNotFoundError(f"Path tidak ditemukan/ambigu: {value}")
    return candidates[0]


with MANIFEST_PATH.open(encoding="utf-8") as handle:
    manifest_rows = list(csv.DictReader(handle))
frame_by_camera = {}
for row in manifest_rows:
    camera_id = Path(row["video_path"]).stem.split("_")[1]
    frame_by_camera.setdefault(camera_id, local_path(row["frame_path"]))
roi_config = load_roi_config(ROI_PATH)
display({camera_id: str(path) for camera_id, path in sorted(frame_by_camera.items())})

{'bogor-kab-bappenda1': 'output\\frames\\atcs_bogor-kab-bappenda1_20261002_185759\\frame_000000.png',
 'bogor-kab-bappenda2': 'output\\frames\\atcs_bogor-kab-bappenda2_20261002_190817\\frame_000000.png',
 'bogor-kab-bappenda3': 'output\\frames\\atcs_bogor-kab-bappenda3_20261002_191842\\frame_000000.png',
 'bogor-kab-pdam2': 'output\\frames\\atcs_bogor-kab-pdam2_20261002_192854\\frame_000000.png',
 'bogor-kab-pemda2': 'output\\frames\\atcs_bogor-kab-pemda2_20261002_193906\\frame_000000.png'}

## Pinpoint setiap kamera

Jalankan sel berikut. Satu jendela interaktif muncul untuk tiap kamera. Jangan menggambar hanya di sekitar lampu; gambar seluruh area lintasan kendaraan yang boleh dipelajari detector.

In [2]:
# Baca ulang dari disk saat cell ini dimulai agar tidak memakai config lama di memory.
roi_config = load_roi_config(ROI_PATH)
pending_config = dict(roi_config)
for camera_id, frame_path in sorted(frame_by_camera.items()):
    with Image.open(frame_path) as image:
        frame_width, frame_height = image.size
    existing_polygon = pending_config.get(camera_id, {}).get("polygon")
    accepted, polygon = pick_polygon_for_camera(frame_path, camera_id, existing_polygon)
    kept_previous = polygon is None
    if kept_previous:
        polygon = existing_polygon or []
    entry = dict(pending_config.get(camera_id, {}))
    entry["roi"] = {"x": 0, "y": 0, "width": frame_width, "height": frame_height}
    entry["polygon"] = polygon
    entry["frame_size"] = {"width": frame_width, "height": frame_height}
    pending_config[camera_id] = entry
    if kept_previous:
        status = "ROI lama dipertahankan"
    elif accepted:
        status = "tersimpan"
    else:
        status = "disimpan dari jendela yang ditutup"
    display(
        f"{camera_id}: {status} {len(polygon)} titik pada {frame_width}x{frame_height}"
    )
# Simpan satu kali setelah semua kamera selesai dipinpoint.
save_roi_config(ROI_PATH, pending_config)
roi_config = pending_config
display(f"ROI config tersimpan setelah semua kamera: {ROI_PATH}")

'bogor-kab-bappenda1: ROI lama dipertahankan 40 titik pada 704x576'

'bogor-kab-bappenda2: ROI lama dipertahankan 41 titik pada 640x480'

'bogor-kab-bappenda3: ROI lama dipertahankan 112 titik pada 640x480'

'bogor-kab-pdam2: ROI lama dipertahankan 41 titik pada 640x360'

'bogor-kab-pemda2: ROI lama dipertahankan 36 titik pada 640x480'

'ROI config tersimpan setelah semua kamera: D:\\Jere\\NewStyle\\output\\roi\\roi_config.json'

## Validasi hasil

Cell ini hanya menampilkan pemeriksaan informatif. Hasil pinpoint tetap sudah disimpan oleh cell sebelumnya.

In [3]:
with ROI_PATH.open(encoding="utf-8") as handle:
    checked_config = json.load(handle)
validation_rows = []
for camera_id, frame_path in sorted(frame_by_camera.items()):
    with Image.open(frame_path) as image:
        width, height = image.size
    entry = checked_config[camera_id]
    polygon = entry.get("polygon", [])
    entry["frame_size"] = {"width": width, "height": height}
    entry["roi"] = {"x": 0, "y": 0, "width": width, "height": height}
    valid = len(polygon) >= 3 and all(
        0 <= point[0] <= width and 0 <= point[1] <= height for point in polygon
    )
    validation_rows.append(
        {
            "camera_id": camera_id,
            "width": width,
            "height": height,
            "vertices": len(polygon),
            "valid": valid,
        }
    )
display(validation_rows)
display("Validasi hanya informatif; hasil pinpoint tidak diblokir.")

[{'camera_id': 'bogor-kab-bappenda1',
  'width': 704,
  'height': 576,
  'vertices': 40,
  'valid': True},
 {'camera_id': 'bogor-kab-bappenda2',
  'width': 640,
  'height': 480,
  'vertices': 41,
  'valid': True},
 {'camera_id': 'bogor-kab-bappenda3',
  'width': 640,
  'height': 480,
  'vertices': 112,
  'valid': True},
 {'camera_id': 'bogor-kab-pdam2',
  'width': 640,
  'height': 360,
  'vertices': 41,
  'valid': True},
 {'camera_id': 'bogor-kab-pemda2',
  'width': 640,
  'height': 480,
  'vertices': 36,
  'valid': True}]

'Validasi hanya informatif; hasil pinpoint tidak diblokir.'